In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression,ElasticNet,LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import VotingClassifier, VotingRegressor, BaggingRegressor, RandomForestClassifier, \
    RandomForestRegressor, AdaBoostRegressor, StackingClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score, log_loss, r2_score
#from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import make_column_selector,make_column_transformer,ColumnTransformer
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from tqdm import tqdm
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier,XGBRegressor
from lightgbm import LGBMClassifier,LGBMRegressor
from catboost import CatBoostClassifier,CatBoostRegressor


In [9]:
housing=pd.read_csv('Housing.csv')
y=housing['price']
X=housing.drop('price',axis=1)
ohe=OneHotEncoder(sparse_output=False,drop='first')
trns = make_column_transformer((ohe, make_column_selector(dtype_include=object)), remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas')
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25)
X_train=trns.fit_transform(X_train)
X_test=trns.transform(X_test)
ss=StandardScaler()

In [10]:
l_rate = np.linspace(0.01, 0.8, 20)
n_est = [50, 100, 200]
depths = [3,5, None]
scores = []
for r in tqdm(l_rate):
    for n in n_est:
        for d in depths:
            gbr = GradientBoostingRegressor(random_state=25, learning_rate=r, n_estimators=n, max_depth=d)
            gbr.fit(X_train, y_train)
            y_pred = gbr.predict(X_test)
            scores.append([r,n,d,r2_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=['Learning Rate','N Estimator','Max Depth','R2 Score'])
df_scores.sort_values(by=['R2 Score'],ascending=False)

  5%|▌         | 1/20 [00:01<00:29,  1.53s/it]


KeyboardInterrupt: 

In [4]:
scores = []
for r in tqdm(l_rate):
    for n in n_est:
        for d in depths:
            gbm = XGBRegressor(random_state=25, learning_rate=r, n_estimators=n, max_depth=d)
            gbm.fit(X_train, y_train)
            y_pred = gbm.predict(X_test)
            scores.append([r,n,d,r2_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=['Learning Rate','N Estimator','Max Depth','R2 Score'])
df_scores.sort_values(by=['R2 Score'],ascending=False)

100%|██████████| 20/20 [00:18<00:00,  1.07it/s]


,Learning Rate,N Estimator,Max Depth,R2 Score
27,0.134737,50,3.0,0.617014
12,0.051579,100,3.0,0.615300
18,0.093158,50,3.0,0.611801
15,0.051579,200,3.0,0.611479
54,0.259474,50,3.0,0.610581
...,...,...,...,...
161,0.716842,200,NaN,0.402892
158,0.716842,100,NaN,0.402768
1,0.010000,50,5.0,0.340971
2,0.010000,50,NaN,0.331190


In [5]:
scores = []
for r in tqdm(l_rate):
    for n in n_est:
        for d in depths:
            gbm = LGBMRegressor(random_state=25, learning_rate=r, n_estimators=n, max_depth=d,verbose=-1)
            gbm.fit(X_train, y_train)
            y_pred = gbm.predict(X_test)
            scores.append([r,n,d,r2_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=['Learning Rate','N Estimator','Max Depth','R2 Score'])
df_scores.sort_values(by=['R2 Score'],ascending=False)

100%|██████████| 20/20 [00:06<00:00,  2.91it/s]


,Learning Rate,N Estimator,Max Depth,R2 Score
14,0.051579,100,NaN,0.645855
13,0.051579,100,5.0,0.641495
19,0.093158,50,5.0,0.638120
20,0.093158,50,NaN,0.638059
12,0.051579,100,3.0,0.631664
...,...,...,...,...
168,0.758421,200,3.0,0.437423
179,0.800000,200,NaN,0.400612
2,0.010000,50,NaN,0.340341
1,0.010000,50,5.0,0.337951


In [6]:
scores = []
for r in tqdm(l_rate):
    for n in n_est:
        for d in depths:
            gbm = CatBoostRegressor(random_state=25, learning_rate=r, n_estimators=n, max_depth=d,verbose=0)
            gbm.fit(X_train, y_train)
            y_pred_prob = gbm.predict(X_test)
            scores.append([r,n,d,r2_score(y_test, y_pred_prob)])
df_scores = pd.DataFrame(scores, columns=['Learning Rate','N Estimator','Max Depth','R2 Score'])
df_scores.sort_values(by=['R2 Score'],ascending=False)

100%|██████████| 20/20 [00:33<00:00,  1.66s/it]


,Learning Rate,N Estimator,Max Depth,R2 Score
55,0.259474,50,5.0,0.642308
73,0.342632,50,5.0,0.642011
76,0.342632,100,5.0,0.638129
31,0.134737,100,5.0,0.638059
28,0.134737,50,5.0,0.635771
...,...,...,...,...
3,0.010000,100,3.0,0.414968
177,0.800000,200,3.0,0.414925
2,0.010000,50,NaN,0.286049
1,0.010000,50,5.0,0.282310


In [12]:
sonar=pd.read_csv('Sonar.csv')
le=LabelEncoder()
y=le.fit_transform(sonar['Class'])
X=sonar.drop('Class',axis=1)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)

In [18]:
 dtc=DecisionTreeClassifier(random_state=25)
 knn=KNeighborsClassifier()
 nb=GaussianNB()
 gbm=GradientBoostingClassifier(random_state=25)
 stack=StackingClassifier([('TREE',dtc),('KNN',knn),('NB',nb)],final_estimator=gbm,passthrough=True)
 stack.fit(X_train,y_train)
 y_pred=stack.predict(X_test)
 print(classification_report(y_test,y_pred))
 y_pred_prob=stack.predict_proba(X_test)
 print(log_loss(y_test,y_pred_prob))

              precision    recall  f1-score   support

           0       0.73      0.94      0.82        34
           1       0.89      0.59      0.71        29

    accuracy                           0.78        63
   macro avg       0.81      0.76      0.76        63
weighted avg       0.80      0.78      0.77        63

0.4828436006761138
